In [ ]:
import json
import os
import re # Import regular expressions for safer replacement

def create_original_english_key(original_entry, line_num_orig):
  """
  Constructs the matching key string from an entry in original.jsonl.
  Handles potential non-string data types and extra whitespace within fields.

  Args:
    original_entry: A dictionary representing a line from original.jsonl.
    line_num_orig: The original line number (1-based) for debugging.

  Returns:
    A string formatted like the 'original_english' field in modified.jsonl,
    or None if required keys are missing or an error occurs.
  """
  required_keys = ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D']
  # Key name in original file (still needed for check)
  orig_index_key_check = "序号\nINDEX"

  # Check for essential text keys first
  if not all(key in original_entry for key in required_keys):
    index_value = original_entry.get(orig_index_key_check, 'N/A')
    # print(f"Debug (Line {line_num_orig} Orig): Skipping entry due to missing required text keys: Index {index_value}") # Less verbose
    return None

  try:
    # --- Convert to string and strip whitespace for each component ---
    story = str(original_entry.get('STORY', '')).strip()
    question = str(original_entry.get('QUESTION', '')).strip()
    # Handle potential None values coming from NaN->null->None conversion
    option_a = str(original_entry.get('OPTION-A', '')).strip()
    option_b = str(original_entry.get('OPTION-B', '')).strip()
    option_c = str(original_entry.get('OPTION-C')).strip() # str(None) is 'None'
    option_d = str(original_entry.get('OPTION-D')).strip()

    # Format the key exactly as in the modified file's 'original_english'
    # Using the individually stripped components
    key = (
        f"###STORY\n{story}\n\n"
        f"###QUESTION\n{question}\n\n"
        f"###OPTIONS\n"
        f"(A) {option_a}\n"
        f"(B) {option_b}\n"
        f"(C) {option_c}\n" # Will correctly format as (C) None if value was None
        f"(D) {option_d}"  # Will correctly format as (D) None if value was None
    )
    return key
  except Exception as e:
      index_value = original_entry.get(orig_index_key_check, 'N/A')
      print(f"Warning: Error converting/formatting key components for Index {index_value} (Orig Line {line_num_orig}): {e}")
      return None


def update_modified_jsonl(original_filepath, modified_filepath, output_filepath):
  """
  Updates the modified JSONL file by adding data fields (ability, index)
  from the original JSONL file based on matching 'original_english' content,
  using English keys for the output. Skips copying the answer.

  Args:
    original_filepath: Path to the original JSONL file.
    modified_filepath: Path to the modified JSONL file.
    output_filepath: Path to save the updated JSONL file.
  """
  original_data_map = {}
  # Field names from original.jsonl (used to retrieve data)
  source_ability_key = "能力\nABILITY"
  source_index_key = "序号\nINDEX"
  # source_answer_key = "答案\nANSWER" # No longer needed

  # Desired field names for the output file
  output_ability_key = "ABILITY"
  output_index_key = "INDEX"
  # output_answer_key = "ANSWER" # No longer needed

  original_line_map = {} # Map key to original line number for debugging

  # --- Step 1: Read original.jsonl and build the map ---
  print(f"Reading original data from: {original_filepath}")
  lines_processed_orig = 0
  lines_error_orig = 0
  try:
    with open(original_filepath, 'r', encoding='utf-8') as f_orig:
      for i, line in enumerate(f_orig):
        line_num_orig = i + 1
        line_content = line.strip()
        if not line_content: continue

        # Replace standalone NaN with null *before* parsing JSON
        line_content = re.sub(r':\s*NaN(\s*[,}])', r': null\1', line_content)

        try:
          entry = json.loads(line_content)
          # Pass line number for debugging context if needed
          key = create_original_english_key(entry, line_num_orig)

          # Check if key was successfully created and the necessary source fields exist
          # Only check for ability and index now
          if key and source_ability_key in entry and source_index_key in entry:
            if key in original_data_map:
                 print(f"Warning: Duplicate key generated from original file line {line_num_orig}. Overwriting entry from line {original_line_map.get(key, 'Unknown')}. Key: '{key[:50]}...'")
            original_data_map[key] = {
                # Store the values needed from the original file using their source keys
                source_ability_key: entry[source_ability_key],
                source_index_key: entry[source_index_key],
                # source_answer_key: entry[source_answer_key] # Don't store answer
            }
            original_line_map[key] = line_num_orig
            lines_processed_orig += 1
          elif key:
             # Key was created, but ability/index fields were missing in the original
             print(f"Warning: Missing source keys ('{source_ability_key}' or '{source_index_key}') in original file line {line_num_orig}. Key: '{key[:50]}...'")
             lines_error_orig += 1
          else:
             lines_error_orig += 1 # Key creation failed
        except json.JSONDecodeError:
          print(f"Warning: Skipping invalid JSON on line {line_num_orig} in {original_filepath}. Content: {line_content[:100]}...")
          lines_error_orig += 1
        except Exception as e:
          print(f"Warning: General error processing line {line_num_orig} in {original_filepath}: {e}")
          lines_error_orig += 1

    print(f"Successfully built map with {len(original_data_map)} entries from {lines_processed_orig} processed lines in original file.")
    if lines_error_orig > 0:
        print(f"Encountered errors or missing data on {lines_error_orig} lines in original file.")

  except FileNotFoundError:
    print(f"Error: Original file not found at {original_filepath}")
    return
  except Exception as e:
    print(f"Error reading original file {original_filepath}: {e}")
    return

  # --- Step 2 & 3: Read modified.jsonl, lookup, and update ---
  updated_entries = []
  matches_found = 0
  matches_not_found = 0
  lines_error_mod = 0
  mismatch_debug_count = 0
  max_mismatch_debug = 0 # Set to 0 to disable mismatch debugging by default

  print(f"Reading and updating modified data from: {modified_filepath}")
  try:
    with open(modified_filepath, 'r', encoding='utf-8') as f_mod:
      for i, line in enumerate(f_mod):
        line_num_mod = i + 1
        line_content = line.strip()
        if not line_content: continue
        try:
          mod_entry = json.loads(line_content)
          original_english_key_text = mod_entry.get('original_english')

          if original_english_key_text:
            # Normalize the key from the modified file
            key_parts = original_english_key_text.split('\n\n')
            normalized_story = ""
            normalized_question = ""
            normalized_options = ""
            if len(key_parts) >= 3:
                normalized_story = key_parts[0].replace("###STORY\n", "").strip()
                normalized_question = key_parts[1].replace("###QUESTION\n", "").strip()
                normalized_options_section = key_parts[2].replace("###OPTIONS\n", "").strip()

                # Split options section into lines, strip each, AND replace 'nan' with 'None'
                options_lines = []
                for opt_line in normalized_options_section.split('\n'):
                    stripped_line = opt_line.strip()
                    # --- Replace 'nan' with 'None' specifically for options C and D ---
                    if stripped_line.lower() == '(c) nan':
                        options_lines.append('(C) None')
                    elif stripped_line.lower() == '(d) nan':
                        options_lines.append('(D) None')
                    else:
                        options_lines.append(stripped_line)
                normalized_options = "\n".join(options_lines)

                # Reconstruct the fully normalized key
                normalized_key = (
                    f"###STORY\n{normalized_story}\n\n"
                    f"###QUESTION\n{normalized_question}\n\n"
                    f"###OPTIONS\n{normalized_options}"
                )
            else:
                # Fallback if splitting failed - use basic normalization
                normalized_key = original_english_key_text.strip().replace('\r\n', '\n')
                print(f"Warning: Could not properly split key components for normalization on Mod Line {line_num_mod}. Using basic strip.")


            # --- Matching Logic ---
            if normalized_key in original_data_map:
              original_info = original_data_map[normalized_key]

              # --- Add/Update fields in the modified entry using OUTPUT keys ---
              # Get values from map using SOURCE keys
              mod_entry[output_ability_key] = original_info[source_ability_key] # Add/Overwrite ability
              mod_entry[output_index_key] = original_info[source_index_key]     # Add/Overwrite index
              # mod_entry[output_answer_key] = original_info[source_answer_key]   # DO NOT Add/Overwrite answer

              updated_entries.append(mod_entry)
              matches_found += 1
            else:
              # --- Mismatch Handling ---
              # Add the entry without the new fields if no match is found
              updated_entries.append(mod_entry)
              matches_not_found += 1

              # --- Mismatch Debugging (Optional and Limited) ---
              if mismatch_debug_count < max_mismatch_debug:
                  print(f"\n--- No Match Found (Mod Line {line_num_mod}) ---")
                  print(f"Key Read/Normalized:\n{normalized_key}\n---")
                  # Try to find a similar key in the original map for comparison
                  found_similar = False
                  partial_key_search = f"###STORY\n{normalized_story}\n\n###QUESTION\n{normalized_question}"
                  for orig_key, orig_line in original_line_map.items():
                      orig_key_parts = orig_key.split('\n\n')
                      if len(orig_key_parts) >= 2:
                          orig_story = orig_key_parts[0].replace("###STORY\n", "").strip()
                          orig_question = orig_key_parts[1].replace("###QUESTION\n", "").strip()
                          if orig_story == normalized_story and orig_question == normalized_question:
                             print(f"Potential Match Key Generated (Orig Line {orig_line}):\n{orig_key}\n---")
                             found_similar = True
                             # break # Uncomment if you only want the first potential match printed
                  if not found_similar:
                      print(f"Debug: No similar key found starting with story/question in original map.")
                  print("-" * 30 + "\n") # Separator
                  mismatch_debug_count += 1

          else:
            print(f"Warning: Missing 'original_english' key on line {line_num_mod} in {modified_filepath}")
            updated_entries.append(mod_entry) # Keep entry as is
            lines_error_mod += 1

        except json.JSONDecodeError:
          print(f"Warning: Skipping invalid JSON on line {line_num_mod} in {modified_filepath}")
          lines_error_mod += 1
        except Exception as e:
          print(f"Warning: Error processing line {line_num_mod} in {modified_filepath}: {e}")
          lines_error_mod += 1

    print(f"Processed modified file: {matches_found} matches found, {matches_not_found} matches not found, {lines_error_mod} lines with errors/missing key.")

  except FileNotFoundError:
    print(f"Error: Modified file not found at {modified_filepath}")
    return
  except Exception as e:
    print(f"Error reading modified file {modified_filepath}: {e}")
    return

  # --- Step 4: Write the updated data ---
  print(f"Writing {len(updated_entries)} updated entries to: {output_filepath}")
  try:
    os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    with open(output_filepath, 'w', encoding='utf-8') as f_out:
      for entry in updated_entries:
        json.dump(entry, f_out, ensure_ascii=False)
        f_out.write('\n')
    print("Successfully updated modified file.")
  except Exception as e:
    print(f"Error writing output file {output_filepath}: {e}")

# --- Define file paths ---
data_directory = 'data'
original_file = os.path.join(data_directory, 'original.jsonl')
modified_file = os.path.join(data_directory, 'modified.jsonl')
output_file = os.path.join(data_directory, 'updated_modified.jsonl')

# --- Run the update process ---
update_modified_jsonl(original_file, modified_file, output_file)


Reading original data from: data/original.jsonl
Successfully built map with 2860 entries from 2860 processed lines in original file.
Reading and updating modified data from: data/modified.jsonl
Processed modified file: 3787 matches found, 0 matches not found, 0 lines with errors/missing key.
Writing 3787 updated entries to: data/updated_modified.jsonl
Successfully updated modified file.
